## Adaptive Cognitive UI and Dialect Translator

Panduan penggunaan model inference:
- XGBoost untuk Adaptive Cognitive UI
- Qwen 3B GGUF via llama-cpp untuk Dialect Translator

## Setup



In [1]:
from google.colab import drive
drive.mount('/content/drive')

"""Base path untuk Google Drive storage"""
base_path = '/content/drive/MyDrive/Kuliah/Lomba/Asean Singapore Hackathon'

datas_path = f'{base_path}/datas'
models_path = f'{base_path}/models'

print(f'datas_path: {datas_path}')
print(f'models_path: {models_path}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
datas_path: /content/drive/MyDrive/Kuliah/Lomba/Asean Singapore Hackathon/datas
models_path: /content/drive/MyDrive/Kuliah/Lomba/Asean Singapore Hackathon/models


In [2]:
!pip install -q xgboost pandas numpy huggingface_hub llama-cpp-python


In [3]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from huggingface_hub import hf_hub_download
from llama_cpp import Llama



## XGBoost Adaptive Cognitive UI

In [4]:
"""Load artifact XGBoost untuk adaptive cognitive UI"""

adaptive_model_path = f'{models_path}/xgboost_classifier.json'
adaptive_params_path = f'{models_path}/deployment_params.json'

if not os.path.isfile(adaptive_model_path):
    raise FileNotFoundError(f'File model tidak ditemukan: {adaptive_model_path}')

if not os.path.isfile(adaptive_params_path):
    raise FileNotFoundError(f'File deployment params tidak ditemukan: {adaptive_params_path}')

with open(adaptive_params_path, 'r') as f:
    adaptive_params = json.load(f)

adaptive_class_names = adaptive_params['class_names']
adaptive_tab_feature_cols = adaptive_params['tab_feature_cols']
adaptive_tab_scaler_mean = np.array(adaptive_params['tab_scaler_mean'], dtype=np.float64)
adaptive_tab_scaler_scale = np.array(adaptive_params['tab_scaler_scale'], dtype=np.float64)

adaptive_model = xgb.XGBClassifier()
adaptive_model.load_model(adaptive_model_path)

print(f'Adaptive classes: {adaptive_class_names}')
print(f'Adaptive expected features: {len(adaptive_tab_feature_cols)}')


Adaptive classes: ['guided', 'simplified', 'standard', 'efficient']
Adaptive expected features: 71


In [5]:
"""Define input contract dan inference function untuk XGBoost adaptive cognitive"""

adaptive_required_features = [
    'hesitation_time_ms',
    'back_button_per_session',
    'rapid_back_sequences',
    'scroll_speed_px_s',
    'scroll_direction_changes',
    'scroll_overshoot_rate',
    'abandon_rate',
    'tap_precision_score',
    'double_tap_rate',
    'time_between_actions_ms',
    'feature_usage_breadth',
    'session_duration_sec',
    'error_encounter_rate',
    'help_button_usage'
]

def adaptive_build_scaled_features(user_sessions):
    """Transform raw user sessions menjadi feature vector XGBoost"""

    sessions_df = pd.DataFrame(user_sessions)

    if sessions_df.empty:
        raise ValueError('user_sessions tidak boleh kosong')

    missing_cols = [col for col in adaptive_required_features if col not in sessions_df.columns]
    if missing_cols:
        raise ValueError(f'Missing required columns: {missing_cols}')

    behavioral_data = sessions_df[adaptive_required_features]

    agg_stats = []
    agg_funcs = ['mean', 'median', 'std', 'min', 'max']

    for feature in adaptive_required_features:
        for stat in agg_funcs:
            if stat == 'std':
                value = behavioral_data[feature].std()
                if pd.isna(value):
                    value = 0.0
            else:
                value = getattr(behavioral_data[feature], stat)()
            agg_stats.append(float(value))

    agg_stats.append(float(len(sessions_df)))

    x = np.array(agg_stats, dtype=np.float64).reshape(1, -1)

    if x.shape[1] != len(adaptive_tab_feature_cols):
        raise ValueError(
            f'Feature size mismatch. Expected {len(adaptive_tab_feature_cols)}, got {x.shape[1]}'
        )

    x_scaled = (x - adaptive_tab_scaler_mean) / adaptive_tab_scaler_scale
    return x_scaled

def predict_adaptive_ui_profile(payload):
    """Input payload -> output profil UI bersih siap pakai"""

    if not isinstance(payload, dict):
        raise ValueError('Payload harus dictionary')

    if 'user_sessions' not in payload:
        raise ValueError('Payload harus mengandung key "user_sessions"')

    x_scaled = adaptive_build_scaled_features(payload['user_sessions'])

    pred_label = int(adaptive_model.predict(x_scaled)[0])
    probs = adaptive_model.predict_proba(x_scaled)[0]

    return {
        'ui_profile': adaptive_class_names[pred_label],
        'label': pred_label,
        'confidence': float(probs[pred_label]),
        'all_probabilities': {
            adaptive_class_names[i]: float(probs[i]) for i in range(len(adaptive_class_names))
        }
    }

print('Adaptive inference helper ready')


Adaptive inference helper ready


## Qwen Dialect Translator


In [6]:
"""Load deployment params translator dan model Qwen 3B GGUF"""

translator_base_path = f'{models_path}/ai-dialect-translator'
translator_config_filename = 'contextual_explainer_params.json'
translator_config_path = f'{translator_base_path}/{translator_config_filename}'

if not os.path.isfile(translator_config_path):
    raise FileNotFoundError(f'Config translator tidak ditemukan: {translator_config_path}')

with open(translator_config_path, 'r', encoding='utf-8') as f:
    translator_params = json.load(f)

translator_gguf_folder = f'{translator_base_path}/3b'
os.makedirs(translator_gguf_folder, exist_ok=True)

translator_gguf_repo = 'Qwen/Qwen2.5-3B-Instruct-GGUF'
translator_gguf_filename = 'qwen2.5-3b-instruct-q4_k_m.gguf'
translator_gguf_path = f'{translator_gguf_folder}/{translator_gguf_filename}'

if not os.path.exists(translator_gguf_path):
    hf_hub_download(
        repo_id=translator_gguf_repo,
        filename=translator_gguf_filename,
        local_dir=translator_gguf_folder,
        local_dir_use_symlinks=False
    )

gguf_params = translator_params['gguf_params']
translator_llm = Llama(
    model_path=translator_gguf_path,
    n_ctx=gguf_params['n_ctx'],
    n_threads=gguf_params['n_threads'],
    n_batch=gguf_params['n_batch'],
    verbose=gguf_params['verbose'],
    n_gpu_layers=0
)

print(f'Translator config loaded: {translator_config_filename}')
print(f'Occupation categories: {len(translator_params["occupation_categories"])}')
print(f'Qwen GGUF loaded: {translator_gguf_filename}')


llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Translator config loaded: contextual_explainer_params.json
Occupation categories: 8
Qwen GGUF loaded: qwen2.5-3b-instruct-q4_k_m.gguf


In [7]:
"""Define helper untuk cleaning output dan inference translator"""

def translator_clean_output(text):
    """Bersihkan artefak format output dari LLM"""

    cleaned = text.strip()
    cleaned = re.sub(r'^(Output|Input)\s*:\s*', '', cleaned, flags=re.IGNORECASE)

    if (cleaned.startswith('"') and cleaned.endswith('"')) or (cleaned.startswith("'") and cleaned.endswith("'")):
        cleaned = cleaned[1:-1].strip()

    cleaned = re.sub(r'^(Output|Input)\s*:\s*', '', cleaned, flags=re.IGNORECASE)

    if (cleaned.startswith('"') and cleaned.endswith('"')) or (cleaned.startswith("'") and cleaned.endswith("'")):
        cleaned = cleaned[1:-1].strip()

    return cleaned

def translate_dialect_explanation(payload):
    """Input payload translator -> output bersih explanation"""

    if not isinstance(payload, dict):
        raise ValueError('Payload harus dictionary')

    required_keys = ['occupation_id', 'concept_text']
    missing_keys = [k for k in required_keys if k not in payload]
    if missing_keys:
        raise ValueError(f'Missing required keys: {missing_keys}')

    occupation_id = payload['occupation_id']
    concept_text = payload['concept_text']

    if occupation_id not in translator_params['occupation_profiles']:
        raise ValueError(f'occupation_id tidak valid: {occupation_id}')

    profile = translator_params['occupation_profiles'][occupation_id]
    sys_prompt = translator_params['system_prompt']
    user_tmpl = translator_params['user_prompt_template']
    gen = translator_params['generation']

    user_msg = user_tmpl.format(
        occupation_label=profile['label'],
        typical_context=profile['typical_context'],
        domain_keywords=', '.join(profile['domain_keywords'][:5]),
        example_concept=profile['example_concept'],
        example_explanation=profile['example_explanation'],
        concept_text=concept_text
    )

    messages = [
        {'role': 'system', 'content': sys_prompt},
        {'role': 'user', 'content': user_msg}
    ]

    start_time = time.time()
    response = translator_llm.create_chat_completion(
        messages=messages,
        max_tokens=gen['max_new_tokens'],
        temperature=gen['temperature'],
        top_p=gen['top_p'],
        top_k=gen['top_k'],
        repeat_penalty=gen['repetition_penalty']
    )
    latency = time.time() - start_time

    raw_text = response['choices'][0]['message']['content'].strip()
    cleaned_text = translator_clean_output(raw_text)

    return {
        'occupation_id': occupation_id,
        'occupation_label': translator_params['occupation_labels'][occupation_id],
        'concept_text': concept_text,
        'explanation': cleaned_text,
        'latency_sec': round(latency, 3)
    }

print('Translator inference helper ready')


Translator inference helper ready


## Demo

Contoh pemanggilan kedua model dan output final masing masing.


In [8]:
"""Demo output untuk adaptive cognitive UI"""

adaptive_payload_example = {
    'user_sessions': [
        {
            'hesitation_time_ms': 1320.0,
            'back_button_per_session': 3,
            'rapid_back_sequences': 1,
            'scroll_speed_px_s': 540.0,
            'scroll_direction_changes': 7,
            'scroll_overshoot_rate': 0.10,
            'abandon_rate': 0.07,
            'tap_precision_score': 0.84,
            'double_tap_rate': 0.03,
            'time_between_actions_ms': 2200.0,
            'feature_usage_breadth': 0.60,
            'session_duration_sec': 310.0,
            'error_encounter_rate': 0.08,
            'help_button_usage': 2
        },
        {
            'hesitation_time_ms': 1470.0,
            'back_button_per_session': 2,
            'rapid_back_sequences': 0,
            'scroll_speed_px_s': 620.0,
            'scroll_direction_changes': 6,
            'scroll_overshoot_rate': 0.09,
            'abandon_rate': 0.05,
            'tap_precision_score': 0.86,
            'double_tap_rate': 0.02,
            'time_between_actions_ms': 1980.0,
            'feature_usage_breadth': 0.63,
            'session_duration_sec': 295.0,
            'error_encounter_rate': 0.06,
            'help_button_usage': 1
        },
        {
            'hesitation_time_ms': 1180.0,
            'back_button_per_session': 4,
            'rapid_back_sequences': 1,
            'scroll_speed_px_s': 510.0,
            'scroll_direction_changes': 9,
            'scroll_overshoot_rate': 0.13,
            'abandon_rate': 0.10,
            'tap_precision_score': 0.81,
            'double_tap_rate': 0.04,
            'time_between_actions_ms': 2380.0,
            'feature_usage_breadth': 0.56,
            'session_duration_sec': 345.0,
            'error_encounter_rate': 0.10,
            'help_button_usage': 3
        }
    ]
}

adaptive_output = predict_adaptive_ui_profile(adaptive_payload_example)

print('Adaptive output:')
print(json.dumps(adaptive_output, indent=2))


Adaptive output:
{
  "ui_profile": "efficient",
  "label": 3,
  "confidence": 0.9995191097259521,
  "all_probabilities": {
    "guided": 0.0001970115990843624,
    "simplified": 6.947117071831599e-05,
    "standard": 0.00021440380078274757,
    "efficient": 0.9995191097259521
  }
}


In [9]:
"""Demo output untuk dialect translator"""

translator_payload_example = {
    'occupation_id': 'fisherman',
    'concept_text': 'Mode offline memungkinkan Anda tetap menggunakan aplikasi tanpa internet. Data disimpan lokal dan tersinkronisasi otomatis.'
}

translator_output = translate_dialect_explanation(translator_payload_example)

print('Translator output:')
print(json.dumps(translator_output, indent=2, ensure_ascii=False))


Translator output:
{
  "occupation_id": "fisherman",
  "occupation_label": "Nelayan",
  "concept_text": "Mode offline memungkinkan Anda tetap menggunakan aplikasi tanpa internet. Data disimpan lokal dan tersinkronisasi otomatis.",
  "explanation": "Penjelasan: \"Seperti perahu nelayan yang bisa berlayar tanpa harus selalu di tepi pantai, mode offline memungkinkan pengguna aplikasi keuangan tetap aktif dan mengumpulkan data hasil tangkapan (keuangan) di perahu (perangkat lokal), sehingga ketika internet kembali, semua data disinkronisasi otomatis. Ini membuat nelayan tidak terganggu saat melaut tanpa harus selalu online.\"",
  "latency_sec": 59.846
}


## Output Contracts

Output final Adaptive Cognitive UI:
- `ui_profile`
- `label`
- `confidence`
- `all_probabilities`

Output final Dialect Translator:
- `occupation_id`
- `occupation_label`
- `concept_text`
- `explanation`
- `latency_sec`